# Stage 2: Liveness Detection Fine-Tuning

**Purpose**: Fine-tune pre-trained VAE on real/fake classification

**Methods**:
1. **Method 1 - Margin Loss**: Push fake reconstruction above margin threshold
2. **Method 2 - Discriminator**: Add latent space classifier

**Dataset**: Balanced real/fake test set from Google Drive

**Hardware**: Google Colab A100 with Mixed Precision Training

**Output**: Fine-tuned models for real/fake classification

## 1. Setup: Mount Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install scipy scikit-learn seaborn tqdm matplotlib -q

In [ ]:
# Extract dataset from zip file
import os
import zipfile

zip_path = '/content/drive/MyDrive/LRdataset/test_balanced_npz.zip'
extract_path = '/content/test_balanced_npz'

print(f"Extracting {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print(f"Extraction complete!")
print(f"Dataset path: {extract_path}")

# Check extracted files
import glob
npz_files = glob.glob(os.path.join(extract_path, '**/*.npz'), recursive=True)
print(f"Found {len(npz_files)} .npz files")

In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n✓ Using Mixed Precision Training (AMP) for A100")
print("  - Models: float32 (stable gradients)")
print("  - Forward pass: float16 (memory & speed)")
print("  - Expected memory savings: ~40-45%")

In [ ]:
%%javascript
// Keep Colab Runtime Alive
function KeepClicking(){
    var cells = document.querySelectorAll('div.cell');
    var currentCell = 0;
    
    setInterval(function(){
        if(cells[currentCell]){
            cells[currentCell].click();
            console.log('Clicked cell ' + currentCell + ' to keep runtime alive');
        }
        currentCell = (currentCell + 1) % cells.length;
    }, 60000); // Every 60 seconds
}

KeepClicking();
console.log('Runtime keeper started - will cycle through cells every 60 seconds');

## 2. Model Definitions

In [ ]:
# Core imports
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy import signal
from scipy.signal import butter, filtfilt
import json
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# TCN Building Block
class TCNBlock(nn.Module):
    """Temporal Convolutional Network Block with residual connection"""
    def __init__(self, C_in, C_out, dilation=1, kernel_size=3):
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        
        self.conv = nn.Conv1d(C_in, C_out, kernel_size=kernel_size, 
                              padding=padding, dilation=dilation)
        self.gn = nn.GroupNorm(1, C_out)
        self.act = nn.SiLU()
        self.res = nn.Conv1d(C_in, C_out, 1) if C_in != C_out else nn.Identity()
    
    def forward(self, x):
        y = self.act(self.gn(self.conv(x)))
        return y + self.res(x)

In [ ]:
# Single-Band VAE
class SingleBandVAE(nn.Module):
    """Feature-Space VAE for one frequency band"""
    def __init__(self, C_in, C_h=48, C_z=12, dilations=[1, 2, 4]):
        super().__init__()
        
        # Encoder
        self.enc_inp = nn.Conv1d(C_in, C_h, 1)
        enc_blocks = [TCNBlock(C_h, C_h, dilation=d) for d in dilations]
        self.encoder = nn.Sequential(*enc_blocks)
        self.enc_out = nn.Conv1d(C_h, C_z * 2, 1)
        
        # Decoder
        self.dec_inp = nn.Conv1d(C_z, C_h, 1)
        dec_blocks = [TCNBlock(C_h, C_h, dilation=d) for d in reversed(dilations)]
        self.decoder = nn.Sequential(*dec_blocks)
        self.dec_out = nn.Conv1d(C_h, C_in, 1)
    
    def encode(self, x):
        h = self.encoder(self.enc_inp(x))
        mu, logvar = torch.chunk(self.enc_out(h), 2, dim=1)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        h = self.decoder(self.dec_inp(z))
        return self.dec_out(h)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

In [ ]:
# Band-Split VAE (3-band architecture)
class BandSplitVAE(nn.Module):
    """Frequency-Decoupled Feature-Space VAE with 3 independent VAEs"""
    def __init__(self, C_in_per_band, C_h=48, C_z=12, dilations=[1, 2, 4]):
        super().__init__()
        
        # 3 independent VAEs
        self.vae_lf = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        self.vae_bp = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        self.vae_hf = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        
        # Learnable fusion weights
        self.fusion_weights = nn.Parameter(torch.ones(3) / 3.0)
    
    def forward(self, x_lf, x_bp, x_hf):
        # Forward through each VAE
        x_hat_lf, mu_lf, logvar_lf = self.vae_lf(x_lf)
        x_hat_bp, mu_bp, logvar_bp = self.vae_bp(x_bp)
        x_hat_hf, mu_hf, logvar_hf = self.vae_hf(x_hf)
        
        # Weighted fusion
        weights = F.softmax(self.fusion_weights, dim=0)
        x_hat_fused = (weights[0] * x_hat_lf + 
                       weights[1] * x_hat_bp + 
                       weights[2] * x_hat_hf)
        
        recons = {'lf': x_hat_lf, 'bp': x_hat_bp, 'hf': x_hat_hf}
        mus = {'lf': mu_lf, 'bp': mu_bp, 'hf': mu_hf}
        logvars = {'lf': logvar_lf, 'bp': logvar_bp, 'hf': logvar_hf}
        
        return recons, mus, logvars, x_hat_fused

In [ ]:
# Discriminator for Method 2
class LatentDiscriminator(nn.Module):
    """
    Discriminator in latent space
    - Input: Concatenated latent vectors from LF, BP, HF bands
    - Output: Binary classification (Real=0, Fake=1)
    """
    def __init__(self, C_z, num_bands=3, hidden_dim=128):
        super().__init__()
        
        # Input: Max + Average pooling for each band
        input_dim = C_z * 2 * num_bands  # C_z * 6
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 1)  # Binary classification
        )
    
    def forward(self, z_lf, z_bp, z_hf):
        # Pool temporal dimension if present (B, C_z, T) -> (B, C_z*2)
        if len(z_lf.shape) == 3:
            z_lf = torch.cat([z_lf.mean(dim=2), z_lf.max(dim=2)[0]], dim=1)
            z_bp = torch.cat([z_bp.mean(dim=2), z_bp.max(dim=2)[0]], dim=1)
            z_hf = torch.cat([z_hf.mean(dim=2), z_hf.max(dim=2)[0]], dim=1)
        
        # Concatenate latent vectors from all bands
        z_concat = torch.cat([z_lf, z_bp, z_hf], dim=1)
        
        # Pass through discriminator
        logits = self.net(z_concat)
        
        return logits

In [ ]:
# Loss Functions
def kl_divergence(mu, logvar):
    """KL divergence for VAE"""
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    return kl.mean()

def band_split_vae_loss(recons, mus, logvars, targets, 
                        x_hat_fused, x_target_full,
                        betas={'lf': 1.0, 'bp': 1.0, 'hf': 1.0},
                        alpha_fusion=0.0):
    """Band-Split VAE Loss: Per-band reconstruction + KL"""
    loss_dict = {}
    total_loss = 0.0
    
    for band in ['lf', 'bp', 'hf']:
        recon_loss = F.l1_loss(recons[band], targets[band])
        kl_loss = kl_divergence(mus[band], logvars[band])
        band_loss = recon_loss + betas[band] * kl_loss
        
        loss_dict[f'recon_{band}'] = recon_loss.item()
        loss_dict[f'kl_{band}'] = kl_loss.item()
        total_loss += band_loss
    
    # Fusion penalty (optional)
    if alpha_fusion > 0 and x_target_full is not None:
        fusion_loss = F.l1_loss(x_hat_fused, x_target_full)
        loss_dict['fusion'] = fusion_loss.item()
        total_loss += alpha_fusion * fusion_loss
    
    loss_dict['total'] = total_loss.item()
    return total_loss, loss_dict

def margin_loss(real_loss, fake_loss, margin=0.5):
    """Margin-based contrastive loss for Method 1"""
    loss_real = real_loss
    loss_fake = torch.clamp(margin - fake_loss, min=0.0)
    return loss_real + loss_fake

## 3. Dataset Definition

In [ ]:
# Stage 2 Dataset
class Stage2Dataset(Dataset):
    """Dataset for Stage 2 with Real + Fake labels"""
    def __init__(self, split_json_path, split_name, T_fixed=300, fps=30,
                 use_acceleration=True, use_angle=True, use_angle_rate=True,
                 fc_low=2.0, fc_high=8.0, filter_order=4, random_crop=True,
                 data_root='/content/test_balanced_npz'):
        self.T_fixed = T_fixed
        self.fps = fps
        self.use_acceleration = use_acceleration
        self.use_angle = use_angle
        self.use_angle_rate = use_angle_rate
        self.fc_low = fc_low
        self.fc_high = fc_high
        self.filter_order = filter_order
        self.random_crop = random_crop
        self.data_root = data_root
        
        # Load split data
        with open(split_json_path, 'r') as f:
            split_data = json.load(f)
        
        if split_name not in split_data:
            raise ValueError(f"split_name must be in JSON, got {split_name}")
        
        split = split_data[split_name]
        
        # Build file index by scanning actual directory structure
        # This handles Korean folder name encoding issues
        print("Building file index from directory...")
        self.file_index = {}  # filename -> full_path
        
        for root, dirs, files in os.walk(data_root):
            for file in files:
                if file.endswith('.npz'):
                    full_path = os.path.join(root, file)
                    self.file_index[file] = full_path
        
        print(f"Found {len(self.file_index)} .npz files in {data_root}")
        
        # Collect files and labels using filename matching
        self.files = []
        self.labels = []
        
        # Real files (label=0)
        missing_real = 0
        for file_path in split['real']:
            filename = os.path.basename(file_path)
            if filename in self.file_index:
                self.files.append(self.file_index[filename])
                self.labels.append(0)
            else:
                missing_real += 1
        
        # Fake files (label=1)
        missing_fake = 0
        for file_path in split['fake']:
            filename = os.path.basename(file_path)
            if filename in self.file_index:
                self.files.append(self.file_index[filename])
                self.labels.append(1)
            else:
                missing_fake += 1
        
        print(f"Butterworth Filter Bank: fc_low={fc_low}Hz, fc_high={fc_high}Hz, order={filter_order}")
        print(f"Loaded {len(self.files)} samples ({len(split['real'])} real, {len(split['fake'])} fake)")
        print(f"  Real files: {len(split['real']) - missing_real}/{len(split['real'])} found")
        print(f"  Fake files: {len(split['fake']) - missing_fake}/{len(split['fake'])} found")
        
        if missing_real > 0 or missing_fake > 0:
            print(f"⚠ Warning: Missing {missing_real} real and {missing_fake} fake files")
        
        if len(self.files) > 0:
            print(f"✓ First file: {os.path.basename(self.files[0])}")
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        # Load npz file
        data = np.load(self.files[idx])
        
        # Combine lips_outer and lips_inner
        lips_outer = data['lips_outer']
        lips_inner = data['lips_inner']
        landmarks = np.concatenate([
            lips_outer[:, :-1, :],
            lips_inner[:, :-1, :]
        ], axis=1)  # (T, 40, 2)
        
        # Normalize by image size
        size = data['size']
        h, w = size
        landmarks = landmarks.copy()
        landmarks[..., 0] /= (w + 1e-8)
        landmarks[..., 1] /= (h + 1e-8)
        
        label = self.labels[idx]
        
        # Build features
        features = self._build_features(landmarks)
        
        # Crop/pad to T_fixed
        features = self._crop_or_pad(features)
        
        # Apply band-split filtering
        x_lf, x_bp, x_hf = self._apply_filters(features)
        
        # Convert to tensors
        x_lf = torch.from_numpy(x_lf).float()
        x_bp = torch.from_numpy(x_bp).float()
        x_hf = torch.from_numpy(x_hf).float()
        label = torch.tensor(label, dtype=torch.long)
        
        return x_lf, x_bp, x_hf, label
    
    def _build_features(self, landmarks):
        """Build feature vector from landmarks"""
        T, N, _ = landmarks.shape
        
        # 1) Position
        position = landmarks
        
        # 2) Velocity
        velocity = np.zeros_like(position)
        velocity[1:] = np.diff(position, axis=0) * self.fps
        
        feature_list = [position, velocity]
        
        # 3) Acceleration
        if self.use_acceleration:
            acceleration = np.zeros_like(position)
            acceleration[1:] = np.diff(velocity, axis=0) * self.fps
            feature_list.append(acceleration)
        
        # 4) Angle and 5) Angle rate
        if self.use_angle or self.use_angle_rate:
            angle = np.arctan2(landmarks[:, :, 1], landmarks[:, :, 0])
            angle = np.expand_dims(angle, axis=-1)
            
            if self.use_angle:
                feature_list.append(angle)
            
            if self.use_angle_rate:
                angle_rate = np.zeros_like(angle)
                angle_rate[1:] = np.diff(angle, axis=0) * self.fps
                feature_list.append(angle_rate)
        
        features = np.concatenate(feature_list, axis=-1)
        return features
    
    def _crop_or_pad(self, features):
        """Crop or pad to T_fixed"""
        T, N, C = features.shape
        
        if T == self.T_fixed:
            return features
        elif T > self.T_fixed:
            if self.random_crop:
                start = np.random.randint(0, T - self.T_fixed + 1)
            else:
                start = 0
            return features[start:start + self.T_fixed]
        else:
            pad_width = ((0, self.T_fixed - T), (0, 0), (0, 0))
            return np.pad(features, pad_width, mode='edge')
    
    def _apply_filters(self, features):
        """Apply Butterworth band-pass filters"""
        T, N, C = features.shape
        nyquist = self.fps / 2.0
        
        # Design filters
        b_lf, a_lf = butter(self.filter_order, self.fc_low / nyquist, btype='low')
        b_bp, a_bp = butter(self.filter_order, [self.fc_low / nyquist, self.fc_high / nyquist], btype='band')
        b_hf, a_hf = butter(self.filter_order, self.fc_high / nyquist, btype='high')
        
        # Apply filters
        x_lf = np.zeros_like(features)
        x_bp = np.zeros_like(features)
        x_hf = np.zeros_like(features)
        
        for n in range(N):
            for c in range(C):
                signal_data = features[:, n, c]
                x_lf[:, n, c] = filtfilt(b_lf, a_lf, signal_data)
                x_bp[:, n, c] = filtfilt(b_bp, a_bp, signal_data)
                x_hf[:, n, c] = filtfilt(b_hf, a_hf, signal_data)
        
        # Reshape to (N*C, T)
        x_lf = x_lf.transpose(1, 2, 0).reshape(N * C, T)
        x_bp = x_bp.transpose(1, 2, 0).reshape(N * C, T)
        x_hf = x_hf.transpose(1, 2, 0).reshape(N * C, T)
        
        # Per-band normalization
        x_lf = (x_lf - x_lf.mean()) / (x_lf.std() + 1e-8)
        x_bp = (x_bp - x_bp.mean()) / (x_bp.std() + 1e-8)
        x_hf = (x_hf - x_hf.mean()) / (x_hf.std() + 1e-8)
        
        return x_lf, x_bp, x_hf

## 4. Configuration

In [ ]:
# Configuration
class Config:
    # Data paths (Google Drive)
    split_json = '/content/drive/MyDrive/LRdataset/data_split (1).json'
    pretrained_model = '/content/drive/MyDrive/LRdataset/stage1_pretrained.pt'
    
    # Data processing
    T_fixed = 300  # 10 seconds @ 30fps
    fps = 30
    
    # Filter Bank
    fc_low = 2.0
    fc_high = 8.0
    filter_order = 4
    
    # Features
    use_acceleration = True
    use_angle = True
    use_angle_rate = True
    K_landmarks = 40
    F_dim = 8  # position + velocity + acceleration + angle + angle_rate
    C_in_per_band = K_landmarks * F_dim  # 320 channels per band
    
    # Model
    C_h = 48
    C_z = 12
    dilations = [1, 2, 4]
    
    # Training (optimized for A100)
    batch_size = 64  # A100 can handle larger batches
    lr = 1e-4  # Lower for fine-tuning
    epochs = 10
    num_workers = 4
    
    # Method 1: Margin Loss
    margin = 0.5
    lambda_margin = 1.0
    
    # Method 2: Discriminator
    lambda_cls = 1.0
    
    # Output
    output_dir = '/content/drive/MyDrive/liveness_checkpoints/stage2'

config = Config()

# Create output directory
os.makedirs(config.output_dir, exist_ok=True)

print("Configuration:")
print(f"  Data split: {config.split_json}")
print(f"  Pre-trained model: {config.pretrained_model}")
print(f"  T_fixed: {config.T_fixed} frames")
print(f"  C_in_per_band: {config.C_in_per_band}")
print(f"  Training: {config.epochs} epochs, batch_size={config.batch_size}")
print(f"  Output: {config.output_dir}")

## 5. Load Datasets

In [ ]:
# Load datasets
print("Loading datasets...")
train_dataset = Stage2Dataset(
    split_json_path=config.split_json,
    split_name='stage2_train',
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
    random_crop=True
)

test_dataset = Stage2Dataset(
    split_json_path=config.split_json,
    split_name='final_test',
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
    random_crop=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True
)

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

## 6. Method 1: Margin Loss Training

In [ ]:
# Training functions for Method 1
@torch.no_grad()
def validate_margin(model, loader, device, margin):
    """Validate with margin loss"""
    model.eval()
    
    real_rec_loss = 0.0
    fake_rec_loss = 0.0
    n_real = 0
    n_fake = 0
    
    for x_lf, x_bp, x_hf, labels in loader:
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        labels = labels.to(device)
        
        real_mask = (labels == 0)
        fake_mask = (labels == 1)
        
        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
        
        if real_mask.sum() > 0:
            real_recons = {k: v[real_mask] for k, v in recons.items()}
            real_mus = {k: v[real_mask] for k, v in mus.items()}
            real_logvars = {k: v[real_mask] for k, v in logvars.items()}
            real_targets = {k: v[real_mask] for k, v in targets.items()}
            
            loss, _ = band_split_vae_loss(
                real_recons, real_mus, real_logvars, real_targets,
                None, None, betas=betas, alpha_fusion=0.0
            )
            real_rec_loss += loss.item()
            n_real += real_mask.sum().item()
        
        if fake_mask.sum() > 0:
            fake_recons = {k: v[fake_mask] for k, v in recons.items()}
            fake_mus = {k: v[fake_mask] for k, v in mus.items()}
            fake_logvars = {k: v[fake_mask] for k, v in logvars.items()}
            fake_targets = {k: v[fake_mask] for k, v in targets.items()}
            
            loss, _ = band_split_vae_loss(
                fake_recons, fake_mus, fake_logvars, fake_targets,
                None, None, betas=betas, alpha_fusion=0.0
            )
            fake_rec_loss += loss.item()
            n_fake += fake_mask.sum().item()
    
    real_rec_loss = real_rec_loss / n_real if n_real > 0 else 0.0
    fake_rec_loss = fake_rec_loss / n_fake if n_fake > 0 else 0.0
    
    real_loss_tensor = torch.tensor(real_rec_loss, device=device)
    fake_loss_tensor = torch.tensor(fake_rec_loss, device=device)
    total_loss = margin_loss(real_loss_tensor, fake_loss_tensor, margin)
    
    return {
        'total': total_loss.item(),
        'real_rec': real_rec_loss,
        'fake_rec': fake_rec_loss,
        'separation': fake_rec_loss - real_rec_loss
    }

def train_epoch_margin(model, loader, optimizer, device, margin, use_amp=True):
    """Train one epoch with margin loss"""
    model.train()
    
    total_loss = 0.0
    real_rec_loss = 0.0
    fake_rec_loss = 0.0
    n_batches = len(loader)
    
    scaler = torch.amp.GradScaler('cuda') if use_amp and torch.cuda.is_available() else None
    
    for x_lf, x_bp, x_hf, labels in tqdm(loader, desc="Training"):
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        real_mask = (labels == 0)
        fake_mask = (labels == 1)
        
        if use_amp and torch.cuda.is_available():
            with torch.amp.autocast('cuda'):
                recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
                targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
                betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
                
                if real_mask.sum() > 0:
                    real_recons = {k: v[real_mask] for k, v in recons.items()}
                    real_mus = {k: v[real_mask] for k, v in mus.items()}
                    real_logvars = {k: v[real_mask] for k, v in logvars.items()}
                    real_targets = {k: v[real_mask] for k, v in targets.items()}
                    
                    real_loss, _ = band_split_vae_loss(
                        real_recons, real_mus, real_logvars, real_targets,
                        None, None, betas=betas, alpha_fusion=0.0
                    )
                else:
                    real_loss = torch.zeros(1, device=device)
                
                if fake_mask.sum() > 0:
                    fake_recons = {k: v[fake_mask] for k, v in recons.items()}
                    fake_mus = {k: v[fake_mask] for k, v in mus.items()}
                    fake_logvars = {k: v[fake_mask] for k, v in logvars.items()}
                    fake_targets = {k: v[fake_mask] for k, v in targets.items()}
                    
                    fake_loss, _ = band_split_vae_loss(
                        fake_recons, fake_mus, fake_logvars, fake_targets,
                        None, None, betas=betas, alpha_fusion=0.0
                    )
                else:
                    fake_loss = torch.zeros(1, device=device)
                
                loss = margin_loss(real_loss, fake_loss, margin)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
            targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
            betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
            
            if real_mask.sum() > 0:
                real_recons = {k: v[real_mask] for k, v in recons.items()}
                real_mus = {k: v[real_mask] for k, v in mus.items()}
                real_logvars = {k: v[real_mask] for k, v in logvars.items()}
                real_targets = {k: v[real_mask] for k, v in targets.items()}
                
                real_loss, _ = band_split_vae_loss(
                    real_recons, real_mus, real_logvars, real_targets,
                    None, None, betas=betas, alpha_fusion=0.0
                )
            else:
                real_loss = torch.zeros(1, device=device)
            
            if fake_mask.sum() > 0:
                fake_recons = {k: v[fake_mask] for k, v in recons.items()}
                fake_mus = {k: v[fake_mask] for k, v in mus.items()}
                fake_logvars = {k: v[fake_mask] for k, v in logvars.items()}
                fake_targets = {k: v[fake_mask] for k, v in targets.items()}
                
                fake_loss, _ = band_split_vae_loss(
                    fake_recons, fake_mus, fake_logvars, fake_targets,
                    None, None, betas=betas, alpha_fusion=0.0
                )
            else:
                fake_loss = torch.zeros(1, device=device)
            
            loss = margin_loss(real_loss, fake_loss, margin)
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
        real_rec_loss += real_loss.item()
        fake_rec_loss += fake_loss.item()
    
    real_rec_loss_avg = real_rec_loss / n_batches
    fake_rec_loss_avg = fake_rec_loss / n_batches
    
    return {
        'total': total_loss / n_batches,
        'real_rec': real_rec_loss_avg,
        'fake_rec': fake_rec_loss_avg,
        'separation': fake_rec_loss_avg - real_rec_loss_avg
    }

In [ ]:
# Train Method 1: Margin Loss
print("="*80)
print("Training Method 1: Margin Loss")
print("="*80)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pre-trained model
print("Loading pre-trained model...")
checkpoint = torch.load(config.pretrained_model, map_location=device, weights_only=False)

model_m1 = BandSplitVAE(
    C_in_per_band=config.C_in_per_band,
    C_h=config.C_h,
    C_z=config.C_z,
    dilations=config.dilations
).to(device)

model_m1.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded checkpoint (val_loss={checkpoint.get('val_loss', 'N/A')})")

# Optimizer
optimizer_m1 = optim.Adam(model_m1.parameters(), lr=config.lr)

# Training loop
print("\nStarting training...")
best_separation = -float('inf')
history_m1 = {'train_loss': [], 'test_loss': [], 'separation': []}

for epoch in range(1, config.epochs + 1):
    epoch_start_time = time.time()
    
    train_loss = train_epoch_margin(model_m1, train_loader, optimizer_m1, device, config.margin, use_amp=True)
    test_loss = validate_margin(model_m1, test_loader, device, config.margin)
    
    epoch_time = time.time() - epoch_start_time
    
    print(
        f"Epoch {epoch}/{config.epochs} - "
        f"train: total={train_loss['total']:.4f}, real={train_loss['real_rec']:.4f}, "
        f"fake={train_loss['fake_rec']:.4f}, sep={train_loss['separation']:.4f} | "
        f"test: total={test_loss['total']:.4f}, real={test_loss['real_rec']:.4f}, "
        f"fake={test_loss['fake_rec']:.4f}, sep={test_loss['separation']:.4f} "
        f"[{epoch_time/60:.1f}min]"
    )
    
    history_m1['train_loss'].append(train_loss['total'])
    history_m1['test_loss'].append(test_loss['total'])
    history_m1['separation'].append(test_loss['separation'])
    
    if test_loss['separation'] > best_separation:
        best_separation = test_loss['separation']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_m1.state_dict(),
            'optimizer_state_dict': optimizer_m1.state_dict(),
            'test_separation': best_separation,
            'config': config
        }, os.path.join(config.output_dir, "method1_margin_best.pt"))
        print(f"  → Saved best model (separation={best_separation:.4f})")

print("="*80)
print(f"Method 1 complete! Best separation: {best_separation:.4f}")
print("="*80)

## 7. Method 2: Discriminator Training

In [ ]:
# Training functions for Method 2
@torch.no_grad()
def validate_discriminator(model, discriminator, loader, device):
    """Validate with discriminator"""
    model.eval()
    discriminator.eval()
    
    total_loss = 0.0
    rec_loss = 0.0
    cls_loss = 0.0
    n_batches = len(loader)
    
    correct = 0
    total = 0
    
    criterion_cls = nn.BCEWithLogitsLoss()
    
    for x_lf, x_bp, x_hf, labels in loader:
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        labels = labels.to(device).float()
        
        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
        
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
        
        loss, loss_dict = band_split_vae_loss(
            recons, mus, logvars, targets,
            x_hat_fused, None, betas=betas, alpha_fusion=0.0
        )
        
        # Discriminator classification
        z_lf = mus['lf']
        z_bp = mus['bp']
        z_hf = mus['hf']
        
        logits = discriminator(z_lf, z_bp, z_hf).squeeze(1)
        cls_loss_batch = criterion_cls(logits, labels)
        
        total_loss_batch = loss + config.lambda_cls * cls_loss_batch
        
        total_loss += total_loss_batch.item()
        rec_loss += loss_dict['total']
        cls_loss += cls_loss_batch.item()
        
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total += labels.size(0)
    
    accuracy = correct / total if total > 0 else 0.0
    
    return {
        'total': total_loss / n_batches,
        'rec': rec_loss / n_batches,
        'cls': cls_loss / n_batches,
        'accuracy': accuracy
    }

def train_epoch_discriminator(model, discriminator, loader, optimizer, device, use_amp=True):
    """Train one epoch with discriminator"""
    model.train()
    discriminator.train()
    
    total_loss = 0.0
    rec_loss = 0.0
    cls_loss = 0.0
    n_batches = len(loader)
    
    correct = 0
    total = 0
    
    criterion_cls = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler('cuda') if use_amp and torch.cuda.is_available() else None
    
    for x_lf, x_bp, x_hf, labels in tqdm(loader, desc="Training"):
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        labels = labels.to(device).float()
        
        optimizer.zero_grad()
        
        if use_amp and torch.cuda.is_available():
            with torch.amp.autocast('cuda'):
                recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
                
                targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
                betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
                
                loss, loss_dict = band_split_vae_loss(
                    recons, mus, logvars, targets,
                    x_hat_fused, None, betas=betas, alpha_fusion=0.0
                )
                
                z_lf = mus['lf']
                z_bp = mus['bp']
                z_hf = mus['hf']
                
                logits = discriminator(z_lf, z_bp, z_hf).squeeze(1)
                cls_loss_batch = criterion_cls(logits, labels)
                
                total_loss_batch = loss + config.lambda_cls * cls_loss_batch
            
            scaler.scale(total_loss_batch).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
            
            targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
            betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
            
            loss, loss_dict = band_split_vae_loss(
                recons, mus, logvars, targets,
                x_hat_fused, None, betas=betas, alpha_fusion=0.0
            )
            
            z_lf = mus['lf']
            z_bp = mus['bp']
            z_hf = mus['hf']
            
            logits = discriminator(z_lf, z_bp, z_hf).squeeze(1)
            cls_loss_batch = criterion_cls(logits, labels)
            
            total_loss_batch = loss + config.lambda_cls * cls_loss_batch
            
            total_loss_batch.backward()
            optimizer.step()
        
        total_loss += total_loss_batch.item()
        rec_loss += loss_dict['total']
        cls_loss += cls_loss_batch.item()
        
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total += labels.size(0)
    
    accuracy = correct / total if total > 0 else 0.0
    
    return {
        'total': total_loss / n_batches,
        'rec': rec_loss / n_batches,
        'cls': cls_loss / n_batches,
        'accuracy': accuracy
    }

In [ ]:
# Train Method 2: Discriminator
print("="*80)
print("Training Method 2: Discriminator")
print("="*80)

# Load pre-trained model again
print("Loading pre-trained model...")
checkpoint = torch.load(config.pretrained_model, map_location=device, weights_only=False)

model_m2 = BandSplitVAE(
    C_in_per_band=config.C_in_per_band,
    C_h=config.C_h,
    C_z=config.C_z,
    dilations=config.dilations
).to(device)

model_m2.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded checkpoint (val_loss={checkpoint.get('val_loss', 'N/A')})")

# Create discriminator
discriminator = LatentDiscriminator(
    C_z=config.C_z,
    num_bands=3,
    hidden_dim=128
).to(device)

n_disc_params = sum(p.numel() for p in discriminator.parameters() if p.requires_grad)
print(f"Discriminator parameters: {n_disc_params:,}")

# Optimizer (both VAE and discriminator)
optimizer_m2 = optim.Adam(
    list(model_m2.parameters()) + list(discriminator.parameters()),
    lr=config.lr
)

# Training loop
print("\nStarting training...")
best_accuracy = 0.0
history_m2 = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(1, config.epochs + 1):
    epoch_start_time = time.time()
    
    train_loss = train_epoch_discriminator(model_m2, discriminator, train_loader, optimizer_m2, device, use_amp=True)
    test_loss = validate_discriminator(model_m2, discriminator, test_loader, device)
    
    epoch_time = time.time() - epoch_start_time
    
    print(
        f"Epoch {epoch}/{config.epochs} - "
        f"train: total={train_loss['total']:.4f}, rec={train_loss['rec']:.4f}, "
        f"cls={train_loss['cls']:.4f}, acc={train_loss['accuracy']:.4f} | "
        f"test: total={test_loss['total']:.4f}, rec={test_loss['rec']:.4f}, "
        f"cls={test_loss['cls']:.4f}, acc={test_loss['accuracy']:.4f} "
        f"[{epoch_time/60:.1f}min]"
    )
    
    history_m2['train_loss'].append(train_loss['total'])
    history_m2['test_loss'].append(test_loss['total'])
    history_m2['train_acc'].append(train_loss['accuracy'])
    history_m2['test_acc'].append(test_loss['accuracy'])
    
    if test_loss['accuracy'] > best_accuracy:
        best_accuracy = test_loss['accuracy']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_m2.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_state_dict': optimizer_m2.state_dict(),
            'test_accuracy': best_accuracy,
            'config': config
        }, os.path.join(config.output_dir, "method2_discriminator_best.pt"))
        print(f"  → Saved best model (accuracy={best_accuracy:.4f})")

print("="*80)
print(f"Method 2 complete! Best accuracy: {best_accuracy:.4f}")
print("="*80)

## 8. Visualization

In [ ]:
# Plot training history for both methods
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Method 1: Margin Loss
axes[0].plot(history_m1['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history_m1['test_loss'], label='Test Loss', marker='s')
axes[0].plot(history_m1['separation'], label='Separation', marker='^')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss / Separation')
axes[0].set_title('Method 1: Margin Loss Training History')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Method 2: Discriminator
ax2 = axes[1]
ax2.plot(history_m2['train_loss'], label='Train Loss', marker='o', color='tab:blue')
ax2.plot(history_m2['test_loss'], label='Test Loss', marker='s', color='tab:orange')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')
ax2.grid(True, alpha=0.3)

ax2_acc = ax2.twinx()
ax2_acc.plot(history_m2['train_acc'], label='Train Accuracy', marker='^', color='tab:green', linestyle='--')
ax2_acc.plot(history_m2['test_acc'], label='Test Accuracy', marker='v', color='tab:red', linestyle='--')
ax2_acc.set_ylabel('Accuracy', color='tab:green')
ax2_acc.tick_params(axis='y', labelcolor='tab:green')
ax2_acc.set_ylim([0, 1])

ax2.set_title('Method 2: Discriminator Training History')
ax2.legend(loc='upper left')
ax2_acc.legend(loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'training_history_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMethod 1 Best Separation: {max(history_m1['separation']):.4f}")
print(f"Method 2 Best Accuracy: {max(history_m2['test_acc']):.4f}")

## Done!

**Training complete!**

**Models saved**:
1. Method 1 (Margin Loss): `method1_margin_best.pt`
2. Method 2 (Discriminator): `method2_discriminator_best.pt`

**Configuration**:
- Mixed Precision Training (AMP) for A100
- Models: float32, Forward pass: float16
- Runtime keeper active

**Next steps**:
1. Download the checkpoints from Google Drive
2. Evaluate on test set
3. Compare performance of both methods